## Imports

In [99]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# PUT GITHUB ROOT PATH HERE
root_dir = "/Users/albert/Library/CloudStorage/OneDrive-andrew.cmu.edu/Documents/CMU 26-27/Undergraduate Research/Victimization/Code/capstone_victimization"
dir = root_dir + "/Data and Resources"
os.chdir(dir)
data = pd.read_csv(os.path.join(dir, "final_data.csv"))
print(data.shape)
print(data.columns)

(8984, 26)
Index(['respondent_id', 'timely_college', 'violent_victimization_18_24',
       'earliest_arrest_before_18', 'total_number_of_incarcerations',
       'r_threatened_to_be_hurt_at_school', 'r_in_a_fight_at_school',
       'days_r_absent_from_school', 'teachers_good',
       'discipline_is_fair_agree_disagree', 'percent_of_peers_who_smoke',
       'percent_of_peers_who_get_drunk_1_times_a_month',
       'percent_of_peers_belong_to_a_gang',
       'percent_of_peers_who_have_had_sex', 'bullying_victim_before_age_12',
       'r_victim_of_bullying_12_18_years', 'sex',
       'r_ever_live_through_hard_times', 'household_net_worth',
       'household_size', 'biological_father_education',
       'biological_mother_education', 'race_ethnicity',
       'asvab_math_verbal_score_percent', 'mother_serve_prison_sentence',
       'father_serve_prison_sentence'],
      dtype='str')


In [100]:
treatment = "timely_college"
outcome = "violent_victimization_18_24"

covariates = [
    col for col in data.columns
    if col not in ["respondent_id", treatment, outcome]
]

X = data[covariates].copy()
Z = data[treatment].copy()
Y = data[outcome].copy()

In [101]:
print(Z.value_counts(dropna=False))
print(Y.value_counts(dropna=False))
for col in covariates:
    print(f"{col}: {X[col].value_counts(dropna=False)}")

timely_college
0.0    4688
1.0    4158
NaN     138
Name: count, dtype: int64
violent_victimization_18_24
0.0    7952
1.0     520
NaN     512
Name: count, dtype: int64
earliest_arrest_before_18: earliest_arrest_before_18
NaN     7408
15.0     305
14.0     295
16.0     285
17.0     283
13.0     172
12.0     119
11.0      61
10.0      22
9.0       13
8.0       11
7.0        5
6.0        3
5.0        2
Name: count, dtype: int64
total_number_of_incarcerations: total_number_of_incarcerations
 0     7972
 1      533
 2      219
 3      105
 4       56
-3       36
 5       33
 6       15
 7        7
 8        6
 11       1
 9        1
Name: count, dtype: int64
r_threatened_to_be_hurt_at_school: r_threatened_to_be_hurt_at_school
 0     7019
 1      767
 2      415
 3      180
 5      122
-4      105
 4       83
 10      75
 20      32
 6       31
 15      24
 7       20
 8       17
 30      17
 12      14
 99      12
 50      10
-2        9
 25       7
 9        5
 17       3
 90       2
 45   

## Remove survey non-responses  

In [102]:
# Replace negative values with NaN for numeric variables
numeric_cols = X.select_dtypes(include="number").columns

X[numeric_cols] = X[numeric_cols].mask(X[numeric_cols] < 0)

# Also remove invalid treatment/outcome values
Z = Z.mask(Z < 0)
Y = Y.mask(Y < 0)

In [103]:
# Propensity scores require a known treatment value
valid_treatment = Z.notna()

X = X.loc[valid_treatment].copy()
Z = Z.loc[valid_treatment].copy()
Y = Y.loc[valid_treatment].copy()

## Separate numeric vs categorical

In [104]:
print(X.columns)

Index(['earliest_arrest_before_18', 'total_number_of_incarcerations',
       'r_threatened_to_be_hurt_at_school', 'r_in_a_fight_at_school',
       'days_r_absent_from_school', 'teachers_good',
       'discipline_is_fair_agree_disagree', 'percent_of_peers_who_smoke',
       'percent_of_peers_who_get_drunk_1_times_a_month',
       'percent_of_peers_belong_to_a_gang',
       'percent_of_peers_who_have_had_sex', 'bullying_victim_before_age_12',
       'r_victim_of_bullying_12_18_years', 'sex',
       'r_ever_live_through_hard_times', 'household_net_worth',
       'household_size', 'biological_father_education',
       'biological_mother_education', 'race_ethnicity',
       'asvab_math_verbal_score_percent', 'mother_serve_prison_sentence',
       'father_serve_prison_sentence'],
      dtype='str')


In [105]:
numeric_features = [
    "earliest_arrest_before_18",
    "total_number_of_incarcerations",
    "days_r_absent_from_school",
    "r_threatened_to_be_hurt_at_school",
    "r_in_a_fight_at_school",
    "household_net_worth",
    "household_size",
    "asvab_math_verbal_score_percent"
]
categorical_features = [
    "teachers_good",
    "discipline_is_fair_agree_disagree",
    "percent_of_peers_who_smoke",
    "percent_of_peers_who_get_drunk_1_times_a_month",
    "percent_of_peers_belong_to_a_gang",
    "percent_of_peers_who_have_had_sex",
    "bullying_victim_before_age_12",
    "r_victim_of_bullying_12_18_years",
    "sex",
    "r_ever_live_through_hard_times",
    "biological_father_education", # 95 = ungraded
    "biological_mother_education", # 95 = ungraded
    "race_ethnicity",
    "mother_serve_prison_sentence",
    "father_serve_prison_sentence"
]

In [106]:
for f in numeric_features:
    if f not in X.columns:
        print(f"Warning: {f} not found in X columns")
    else:
        print(f)

earliest_arrest_before_18
total_number_of_incarcerations
days_r_absent_from_school
r_threatened_to_be_hurt_at_school
r_in_a_fight_at_school
household_net_worth
household_size
asvab_math_verbal_score_percent


In [107]:
for f in categorical_features:
    if f not in X.columns:
        print(f"Warning: {f} not found in X columns")
    else:
        print(f)

teachers_good
discipline_is_fair_agree_disagree
percent_of_peers_who_smoke
percent_of_peers_who_get_drunk_1_times_a_month
percent_of_peers_belong_to_a_gang
percent_of_peers_who_have_had_sex
bullying_victim_before_age_12
r_victim_of_bullying_12_18_years
sex
r_ever_live_through_hard_times
biological_father_education
biological_mother_education
race_ethnicity
mother_serve_prison_sentence
father_serve_prison_sentence


## Preprocess for logistic regression
Impute numerical with median, and impute categorical with mode.
Also one-hot encode categoricals

In [108]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

## Create propensity score model

In [109]:
ps_model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic", LogisticRegression(max_iter=1000))
])

## Estimate propensity scores

In [110]:
ps_model.fit(X, Z)

propensity_scores = ps_model.predict_proba(X)[:, 1]

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [111]:
data_ps = data.loc[valid_treatment].copy()

data_ps["propensity_score"] = propensity_scores

data_ps[[
    treatment,
    "propensity_score"
]].head()

,timely_college,propensity_score
0,0.0,0.611648
1,1.0,0.617617
2,0.0,0.385448
3,1.0,0.418487
4,0.0,0.407878


## Propensity Score Distributions

In [118]:
import plotly.express as px

fig = px.histogram(
    data_ps,
    x="propensity_score",
    color=treatment,
    nbins=100,
    barmode="overlay",
    opacity=0.6,
    title="Propensity Score Distribution by Treatment Group",
)

fig.update_layout(
    xaxis_title="Propensity Score",
    yaxis_title="Count",
    width=800
)

fig.show()

## LR Coefficients

In [ ]:
pd.DataFrame({
    'Feature': ps_model["logistic"].feature_names_in_,
    'Coefficient': ps_model["logistic"].coef_[0]  # .coef_[0] extracts the 1D array for binary classification
})

array([[-3.45753697e-02, -6.37292791e-01, -4.15487614e-02,
        -1.48367305e-02, -2.18767995e-03,  1.82429698e-06,
        -6.45754487e-02,  2.31032769e-02,  1.42601740e-01,
         6.63943145e-02, -1.96889403e-01, -5.91320600e-02,
         1.02988461e-01, -4.03287320e-02, -4.94293288e-02,
        -6.02558088e-02,  1.09476983e-02,  8.17202773e-02,
         3.07417804e-02, -6.70729811e-02, -1.03362183e-01,
        -3.39871564e-02, -2.64209027e-02,  1.16349912e-02,
         5.02956955e-02, -4.85480365e-02,  8.69298216e-02,
         1.34178380e-02, -6.84258772e-02, -3.23190085e-02,
        -4.66281807e-02,  4.91310812e-02,  1.22310914e-01,
        -8.65197986e-02, -7.65934519e-02, -5.53541525e-02,
         5.84766958e-02, -1.05502104e-01,  2.97563253e-03,
        -5.00010442e-02, -2.21827208e-01,  1.74801798e-01,
         5.24394145e-02, -9.94648186e-02, -5.33232173e-04,
        -6.31926998e-03,  6.38064117e-03, -1.54993587e-02,
        -1.25815837e-02, -1.47438272e-02, -3.15984789e-0